In [ ]:
import os
import requests
import math
from PIL import Image
import io
import pandas as pd
import geopandas as gpd
import pickle
import time
from shapely.geometry import Point


class AdaptiveSatelliteImageCollector:
    def __init__(self, csv_path, chicago_shapefile_path, illinois_shapefile_path,
                 image_resolution=1024, checkpoint_file='point_progress_checkpoint.pkl'):
        """

        Args:
            csv_path (str): CSV: lat, lon, point_id
            chicago_shapefile_path (str): Chicago Census Tract shapefile
            illinois_shapefile_path (str): Illinois Census Tract shapefile
            image_resolution (int)
            checkpoint_file (str)
        """
        self.checkpoint_file = checkpoint_file
        self.processed_ids = set()
        self.load_checkpoint()


        try:
            self.points_df = pd.read_csv(csv_path, sep='\t')
            if len(self.points_df.columns) <= 1:
                self.points_df = pd.read_csv(csv_path, sep=',')
        except Exception:
            self.points_df = pd.read_csv(csv_path)

        self.points_df.columns = self.points_df.columns.str.strip()

        print(f"Loaded {len(self.points_df)} points from {csv_path}")
        print(f"Columns: {list(self.points_df.columns)}\n")

        # 读取 Chicago Census Tract shapefile
        print(f"Loading Chicago Census Tract shapefile: {chicago_shapefile_path}")
        try:
            self.chicago_gdf = gpd.read_file(chicago_shapefile_path)
            print(f"  ✓ Loaded {len(self.chicago_gdf)} Chicago census tracts")
            print(f"    CRS: {self.chicago_gdf.crs}")
        except Exception as e:
            print(f"  ✗ Error loading Chicago shapefile: {e}")
            self.chicago_gdf = None

        #  Illinois Census Tract shapefile
        print(f"\nLoading Illinois Census Tract shapefile: {illinois_shapefile_path}")
        try:
            self.illinois_gdf = gpd.read_file(illinois_shapefile_path)
            print(f"  ✓ Loaded {len(self.illinois_gdf)} Illinois census tracts")
            print(f"    CRS: {self.illinois_gdf.crs}")
        except Exception as e:
            print(f"  ✗ Error loading Illinois shapefile: {e}")
            self.illinois_gdf = None

        self.image_resolution = image_resolution


        self.output_dir = 'satellite_images_adaptive1'
        os.makedirs(self.output_dir, exist_ok=True)


        self.log_file = os.path.join(self.output_dir, 'adaptive_sizes_log1.csv')

        print(f"\nAdaptive Satellite Image Configuration:")
        print(f"  Image resolution: {image_resolution}×{image_resolution} px")
        print(f"  Output directory: ./{self.output_dir}/")
        print(f"  Log file: ./{self.log_file}\n")

    # ================================================================= #
    #  Checkpoint
    # ================================================================= #
    def load_checkpoint(self):

        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'rb') as f:
                    checkpoint = pickle.load(f)
                    self.processed_ids = checkpoint.get('processed_ids', set())
                    print(f"Restored checkpoint: {len(self.processed_ids)} points already processed\n")
            except Exception as e:
                print(f"Failed to load checkpoint: {e}\n")
                self.processed_ids = set()
        else:
            self.processed_ids = set()

    def save_checkpoint(self, point_id):

        try:
            self.processed_ids.add(point_id)
            checkpoint = {'processed_ids': self.processed_ids}
            with open(self.checkpoint_file, 'wb') as f:
                pickle.dump(checkpoint, f)
        except Exception as e:
            print(f"Failed to save checkpoint: {e}")

    # ================================================================= #
    #  Census Tract
    # ================================================================= #
    def get_census_tract_size(self, lat, lon):
        """


        Args:
            lat (float):
            lon (float):

        Returns:
            dict
        """
        try:

            point = Point(lon, lat)

            # Chicago shapefile
            if self.chicago_gdf is not None:
                for idx, tract in self.chicago_gdf.iterrows():
                    if tract.geometry.contains(point):
                        return self._extract_tract_info(tract, self.chicago_gdf, source='Chicago')

            # Illinois shapefile
            if self.illinois_gdf is not None:
                for idx, tract in self.illinois_gdf.iterrows():
                    if tract.geometry.contains(point):
                        return self._extract_tract_info(tract, self.illinois_gdf, source='Illinois')

            #
            print(f"    ⚠ Point ({lat}, {lon}) not found in Chicago or Illinois Census Tracts, using default 0.5 mile")
            return {
                'tract_found': False,
                'source': 'Default',
                'area_m2': None,
                'area_km2': None,
                'distance_m': 0.5 * 1609.34,  # 默认 0.5 mile
                'tract_id': None
            }

        except Exception as e:
            print(f"    ⚠ Error getting Census Tract info: {e}")
            return {
                'tract_found': False,
                'source': 'Error',
                'area_m2': None,
                'area_km2': None,
                'distance_m': 0.5 * 1609.34,
                'tract_id': None
            }

    def _extract_tract_info(self, tract, gdf, source=''):


        if gdf.crs.is_geographic:
            tract_proj = gpd.GeoDataFrame([tract], crs=gdf.crs).to_crs('epsg:3857')
            area_m2 = tract_proj.geometry.area.iloc[0]
        else:
            area_m2 = tract.geometry.area

        area_km2 = area_m2 / 1_000_000


        side_length_m = math.sqrt(area_m2)

        # tract ID
        tract_id = None
        for col in ['GEOID', 'GEOID20', 'GEOID10', 'TRACT', 'NAME', 'TRACTCE', 'TRACTCE20']:
            if col in tract.index:
                tract_id = tract[col]
                break

        return {
            'tract_found': True,
            'source': source,
            'area_m2': area_m2,
            'area_km2': area_km2,
            'distance_m': side_length_m,
            'tract_id': tract_id
        }

    # ================================================================= #
    #  Bounds
    # ================================================================= #
    def get_tile_bounds(self, lat, lon, distance_m):

        lat_offset = distance_m / 111320.0
        lat_half = lat_offset / 2


        lon_offset = lat_offset / math.cos(math.radians(lat))
        lon_half = lon_offset / 2

        return {
            'north': round(lat + lat_half, 6),
            'south': round(lat - lat_half, 6),
            'east': round(lon + lon_half, 6),
            'west': round(lon - lon_half, 6),
            'distance_m': distance_m
        }


    def download_satellite_image(self, lat, lon, point_id, census_info, mapbox_token):

        distance_m = census_info['distance_m']
        bounds = self.get_tile_bounds(lat, lon, distance_m)
        bbox = f"[{bounds['west']},{bounds['south']},{bounds['east']},{bounds['north']}]"

        half_res = self.image_resolution // 2  # 512
        url = (
            f"https://api.mapbox.com/styles/v1/mapbox/satellite-v9/static/"
            f"{bbox}/"
            f"{half_res}x{half_res}"
            f"@2x?access_token={mapbox_token}"
        )

        try:
            response = requests.get(url, timeout=30)
            response.raise_for_status()

            image = Image.open(io.BytesIO(response.content))


            tract_suffix = f"_tract{census_info['tract_id']}" if census_info['tract_id'] else ""
            output_path = os.path.join(
                self.output_dir,
                f"point{point_id}{tract_suffix}_{round(lat, 6)}_{round(lon, 6)}.png"
            )
            image.save(output_path)

            print(f"  ✓ Downloaded: {image.size[0]}×{image.size[1]} px")
            print(f"    Coverage: {distance_m:.1f}m × {distance_m:.1f}m")
            if census_info['area_km2']:
                print(f"    Census Tract area: {census_info['area_km2']:.2f} km²")
            print(f"    Source: {census_info['source']}")

            return output_path, distance_m

        except Exception as e:
            print(f"  ✗ Download error: {e}")
            return None, None


    def process_point(self, lat, lon, point_id, mapbox_token):

        lat = round(lat, 6)
        lon = round(lon, 6)

        print(f"Processing point {point_id} at ({lat}, {lon})")


        print(f"  Getting Census Tract info...")
        census_info = self.get_census_tract_size(lat, lon)

        if census_info['tract_found']:
            print(f"  ✓ Found Census Tract:")
            print(f"    Source: {census_info['source']}")
            print(f"    ID: {census_info['tract_id']}")
            print(f"    Area: {census_info['area_km2']:.2f} km²")


        print(f"  Downloading satellite image...")
        satellite_path, distance_m = self.download_satellite_image(
            lat, lon, point_id, census_info, mapbox_token
        )

        result = {
            'point_id': point_id,
            'lat': lat,
            'lon': lon,
            'satellite_path': satellite_path,
            'tract_id': census_info['tract_id'],
            'tract_found': census_info['tract_found'],
            'tract_source': census_info['source'],
            'tract_area_km2': census_info['area_km2'],
            'coverage_distance_m': distance_m
        }


        self.save_checkpoint(point_id)

        return result


    def process_all_points(self, mapbox_token):

        results = []
        total = len(self.points_df)

        for i, (_, row) in enumerate(self.points_df.iterrows()):
            point_id = int(row['point_id'])
            lat = float(row['lat'])
            lon = float(row['lon'])

            print(f"\n{'='*70}")
            print(f"Point {point_id}  ({i + 1}/{total})")
            print(f"{'='*70}")


            if point_id in self.processed_ids:
                print(f"  (Already processed, skipped)")
                continue

            result = self.process_point(lat, lon, point_id, mapbox_token)
            results.append(result)


            time.sleep(1.0)


        self.save_results(results)
        return results


    def save_results(self, results):
        """保存所有结果到CSV"""
        try:
            data_rows = []

            for r in results:
                data_rows.append({
                    'point_id': r['point_id'],
                    'lat': r['lat'],
                    'lon': r['lon'],
                    'satellite_image_path': r.get('satellite_path', ''),
                    'tract_found': r['tract_found'],
                    'tract_source': r['tract_source'],
                    'tract_id': r['tract_id'],
                    'tract_area_km2': r['tract_area_km2'],
                    'coverage_distance_m': r['coverage_distance_m']
                })

            if data_rows:
                df = pd.DataFrame(data_rows)
                output_csv = os.path.join(self.output_dir, 'satellite_index.csv')
                df.to_csv(output_csv, index=False)
                print(f"\n✓ Saved satellite image index: {len(data_rows)} points")
                print(f"  → {output_csv}")


                found_count = df['tract_found'].sum()
                print(f"\n✓ Statistics:")
                print(f"  Total points: {len(df)}")
                print(f"  Found in Census Tract: {found_count}")
                print(f"  Not found: {len(df) - found_count}")

                if found_count > 0:
                    chicago_count = (df['tract_source'] == 'Chicago').sum()
                    illinois_count = (df['tract_source'] == 'Illinois').sum()
                    print(f"\n  Breakdown:")
                    print(f"    Chicago Census Tracts: {chicago_count}")
                    print(f"    Illinois Census Tracts: {illinois_count}")

                if 'coverage_distance_m' in df.columns:
                    valid_distances = df[df['coverage_distance_m'].notna()]['coverage_distance_m']
                    if len(valid_distances) > 0:
                        print(f"\n  Coverage distance statistics:")
                        print(f"    Mean: {valid_distances.mean():.1f}m")
                        print(f"    Min: {valid_distances.min():.1f}m")
                        print(f"    Max: {valid_distances.max():.1f}m")

        except Exception as e:
            print(f"Error saving results: {e}")



def main():
    # -------- 配置 --------
    csv_path = './od_unique_coordinates_detailed.csv'
    chicago_shapefile_path = './Census_Tracts_20260408/geo_export_a7429ffe-7e27-44b3-8461-6924a49f7682.shp'
    illinois_shapefile_path = './tl_2025_17_tract/tl_2025_17_tract.shp'  # ← Illinois shapefile
    mapbox_token = ''


    collector = AdaptiveSatelliteImageCollector(
        csv_path=csv_path,
        chicago_shapefile_path=chicago_shapefile_path,
        illinois_shapefile_path=illinois_shapefile_path,
        image_resolution=1024,
        checkpoint_file='point_progress_checkpoint3.pkl'
    )


    results = collector.process_all_points(mapbox_token)


    print(f"\n{'='*70}")
    print(f"Adaptive satellite image download complete!")
    print(f"Total points processed: {len(results)}")
    print(f"Output directory: ./{collector.output_dir}/")
    print(f"Index file: ./{collector.output_dir}/satellite_index.csv")
    print(f"{'='*70}")


if __name__ == "__main__":
    main()

Loaded 1148 points from ./od_unique_coordinates_detailed.csv
Columns: ['lat', 'lon', 'point_id', 'n_as_origin', 'n_as_dest', 'n_total']

Loading Chicago Census Tract shapefile: ./Census_Tracts_20260408/geo_export_a7429ffe-7e27-44b3-8461-6924a49f7682.shp
  ✓ Loaded 878 Chicago census tracts
    CRS: EPSG:4326

Loading Illinois Census Tract shapefile: ./tl_2025_17_tract/tl_2025_17_tract.shp
  ✓ Loaded 3265 Illinois census tracts
    CRS: EPSG:4269

Adaptive Satellite Image Configuration:
  Image resolution: 1024×1024 px
  Output directory: ./satellite_images_adaptive1/
  Log file: ./satellite_images_adaptive1/adaptive_sizes_log1.csv


Point 1  (1/1148)
Processing point 1 at (41.891022, -87.612931)
  Getting Census Tract info...
  ✓ Found Census Tract:
    Source: Chicago
    ID: None
    Area: 2.23 km²
  ✓ Downloaded: 1024×1024 px
    Coverage: 1494.4m × 1494.4m
    Census Tract area: 2.23 km²
    Source: Chicago

Point 2  (2/1148)
Processing point 2 at (41.928424, -87.684906)
  Getting 

In [ ]:
import os


project_dir = '/blue/shenhaowang/qingqisong/be-and-active-travel'
os.chdir(project_dir)